In [1]:
Year=110
Period='AP'
discount=(2524393+4008113+166767)/(2921037+4300427+153469)

import geopandas as gpd
import os
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import networkx as nx
from shapely.geometry import Point
from scipy.optimize import fsolve
import time

In [2]:
#########################################
# 讀取 Shapefile 路網和節點資料
net= 'real' # 'sim' or 'real'
if net == 'sim':
    road_network = gpd.read_file('Net_V3.shp')
    #remove the link that Level exceeds 6 and the virtual link that Level is 12
    road_network = road_network[road_network['LEVEL'] <= 6]
    road_network_virtual = gpd.read_file('Net_V3.shp')
    road_network_virtual = road_network_virtual[road_network_virtual['LEVEL'] == 12]
    #centroids connectors
    road_network_centroid = gpd.read_file('Net_V3.shp')
    road_network_centroid = road_network_centroid[road_network_centroid['LEVEL'] == 11]
    # combine the link and the centroid
    road_network = pd.concat([road_network, road_network_centroid, road_network_virtual], ignore_index=True)
elif net == 'real':
    road_network = gpd.read_file('Y110Net.shp')
    road_network = road_network[road_network['LEVEL'] <= 6]
    road_network_centroid = gpd.read_file('Y110Net.shp')
    road_network_centroid = road_network_centroid[road_network_centroid['LEVEL'] == 11]
    road_network = pd.concat([road_network, road_network_centroid], ignore_index=True)

#for i in road_network['FFTIME'], if nan, set it to 0
# road_network['FFTIME'] = road_network['FFTIME'].fillna(0)

zone_data = gpd.read_file('TZone601.shp')
zone_data = zone_data[zone_data['ID'] <=571]
zone_data = zone_data.sort_values(by=['TAZONENUM'])
zone_data.set_index('TAZONENUM', inplace=True)
#set the TAZ number as the index
TAZONENUM = list(set(zone_data.index.tolist()))

Nodes = []
for i in road_network['A'].values:
    if i not in Nodes:
        Nodes.append(i)
for i in road_network['B'].values:
    if i not in Nodes:
        Nodes.append(i)
Nodes.sort()

# 初始化有向圖
G = nx.DiGraph()

# 將邊加入圖中
for _, row in road_network.iterrows():
    G.add_edge(
        row['A'], row['B'],
        FFTIME=row['FFTIME'],
        CAPACITY=row['CAPACITY'],
        SPEED=row['SPEED'],
        LENGTH=row['LENGTH'],
        ALPHA=row['ALPHA'],
        BETA=row['BETA'],
        # toll=row['TOLL'],
        flow=0  # 初始化流量為0
    )

for u, v, data in G.edges(data=True):
    # 確保 FFSPEED 不為零
    if data['SPEED'] == 0:
        data['SPEED'] = 50  # 設定預設自由流速度（km/h）
    # 根據 FFSPEED 和長度計算 t0
    data['FFTIME'] = data['LENGTH'] / data['SPEED'] * 60  # 轉換為分鐘
    # 確保 t0 不為零
    if data['FFTIME'] == 0:
        data['FFTIME'] = 0.0001  # 設定一個很小的正數
################################################

def read_OD_matrix(year, period):
    file=f'OD_{year}_{period}.csv'
    All_np=np.loadtxt(file,delimiter=',')
    return All_np

OD=read_OD_matrix(Year, Period)
OD=pd.DataFrame(OD,index=TAZONENUM,columns=TAZONENUM)*discount

#read TAZ.csv to get the zone id
TAZ_list = pd.read_csv('TAZ.csv')
#make a dictionary to store the TAZONENUM and the corresponding ID<=600 in TAZ
tazonenum = {}
for i in range(len(TAZ_list)):
    if int(TAZ_list['ID'][i])<=571:
        tazonenum[TAZ_list['TAZONENUM'][i]] = TAZ_list['ID'][i]
######################################################
# 確認 OD 矩陣中的起訖對是否都對應到分區中的區心節點
od_assignments = []
trip_threshold = 1

# 確認 od_assignments_{Year}_{Period}_{trip_threshold}.csv 是否存在，若存在則讀取檔案，否則重新建立
if os.path.exists(f'od_assignments_{Year}_{Period}_{trip_threshold}.csv'):
    # 讀取 od_assignments 檔案
    od_assignments = pd.read_csv(f'od_assignments_{Year}_{Period}_{trip_threshold}.csv').values.tolist()
    for i in range(len(od_assignments)):
        od_assignments[i][0] = int(od_assignments[i][0])
        od_assignments[i][1] = int(od_assignments[i][1])
else:
    # use od and taz61 to get the origin and destination node
    for origin_zone in OD.index:
        for dest_zone in OD.columns:
            origin_node = tazonenum[origin_zone]
            dest_node = tazonenum[dest_zone]
            trips = OD.at[origin_zone, dest_zone]
            # 建立起訖對的旅次指派清單，若旅次量大於門檻值才加入
            
            if trips >= trip_threshold:
                split_threshold = 10         #當起訖對旅次大於此數值時，將旅次拆分
                if trips < split_threshold:
                    od_assignments.append([origin_node, dest_node, trips])
                else:
                    trip_int = int(trips)
                    for split in range(trip_int // split_threshold):
                        od_assignments.append([origin_node, dest_node, split_threshold])
                    if trip_int % split_threshold != 0:  
                        od_assignments.append([origin_node, dest_node, trips%split_threshold])
        # shuffling the list
    np.random.shuffle(od_assignments)

    # 匯出 od_assignments 為 csv 檔，命名包含threshold的檔案
    od_assignments_df = pd.DataFrame(od_assignments, columns=['origin_node', 'dest_node', 'trips'])
    od_assignments_df.to_csv(f'od_assignments_{Year}_{Period}_{trip_threshold}.csv', index=False)

# 確認輸出資料
# print("起訖對旅次分配:")
# for origin_node, dest_node, trips in od_assignments[110:120]:  # 僅顯示 10 筆
    # print(f"起點: {origin_node}, 終點: {dest_node}, 旅次量: {trips}")
print(f"總共 {len(od_assignments)} 筆旅次分配")

總共 147142 筆旅次分配


In [3]:
start_time = time.time()

# 初始化道路參數
for u, v, data in G.edges(data=True):
    data['t0'] = data['FFTIME']  # 調整過的自由流時間（分鐘）
    data['capacity'] = data['CAPACITY']
    data['alpha'] = data['ALPHA']
    data['beta'] = data['BETA']
    data['length'] = data['LENGTH']
    data['flow'] = 0  # 初始流量
    data['time'] = data['t0']  # 初始旅行時間

# 定義 BPR 函數來更新旅行時間
def update_travel_times(G):
    for u, v, data in G.edges(data=True):
        volume = data['flow']
        capacity = data['capacity']
        alpha = data['alpha']
        beta = data['beta']
        t0 = data['t0']
        data['time'] = t0 * (1 + alpha * (volume / capacity) ** beta)

max_iterations = 1   # 最大迭代次數
update_freqency = 100  # 每執行幾次指派後要更新旅行時間

for origin_node, dest_node, trips in od_assignments:
    if trips > 0:
        try:
            path = nx.bellman_ford_path(G, source=origin_node, target=dest_node, weight='time')
            # 將流量分配到路徑中的每一條邊
            for i in range(len(path) - 1):
                u, v = path[i], path[i + 1]
                G[u][v]['flow'] += trips
        except nx.NetworkXNoPath:
            print(f"無法在原點 {origin_node} 與目的地 {dest_node} 之間找到路徑")
            continue  # 無法在原點與目的地之間找到路徑
        update_travel_times(G)

# 將結果寫回 shapefile
for u, v, data in G.edges(data=True):
    if data['capacity'] > 0:
        data['v_over_c'] = data['flow'] / data['capacity']
    else:
        data['v_over_c'] = 0  # 或設定為預設值
    data['assigned_speed'] = data['length'] / (data['time'] / 60)  # 計算指派後的速度（km/h）

In [4]:
# 從 NetworkX 圖形中提取邊的屬性
edge_data = []

for u, v, data in G.edges(data=True):
    edge_data.append({
        'A': u,
        'B': v,
        'flow': data['flow'],
        'v_over_c': data['v_over_c'],
        'assigned_speed': data['assigned_speed'],
        'time': data['time']
    })

# 將邊的資料轉換為 DataFrame
edge_df = pd.DataFrame(edge_data)

# 假設 road_network 是您的原始路網 GeoDataFrame，包含 'A', 'B' 和 'geometry' 欄位
# 如果欄位名稱不同，請修改 'on' 參數中的欄位名稱

# 合併資料
merged_gdf = road_network.merge(edge_df, on=['A', 'B'], how='left')

# 設定 CRS（如果尚未設定）
merged_gdf.crs = road_network.crs

# 輸出檔案名稱
output_file = f'Complete_Assignment_Y{Year}_{Period}_{max_iterations}iter_PT30_demolition_pop_revised.shp'

# 將結果寫入 Shapefile
merged_gdf.to_file(output_file, driver='ESRI Shapefile')
print(f"結果已輸出至 {output_file}")

C:\Users\chiuj\AppData\Local\Temp\ipykernel_20680\1600816173.py:30: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  merged_gdf.to_file(output_file, driver='ESRI Shapefile')
C:\Users\chiuj\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'assigned_speed' to 'assigned_s'
  ogr_write(


結果已輸出至 Complete_Assignment_Y110_AP_1iter_PT30_demolition_pop_revised.shp
